<a href="https://colab.research.google.com/github/sampada-2006/AI-Music-Source-Separation/blob/main/FluetFineTunningColab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install -q dora-search einops "julius>=0.2.3" "lameenc>=1.2" openunmix pyyaml soundfile tqdm
!pip install -q --no-deps git+https://github.com/facebookresearch/demucs#egg=demucs

  Preparing metadata (setup.py) ... done


In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
#Project Setup & Synthetic Data Generation
import os
import torch
import soundfile as sf

BASE_DIR = '/content/drive/MyDrive/Demucs_Flute_Project'
DATASET_DIR = os.path.join(BASE_DIR, 'dataset')
CHECKPOINT_DIR = os.path.join(BASE_DIR, 'checkpoints')

os.makedirs(os.path.join(DATASET_DIR, 'train'), exist_ok=True)
os.makedirs(os.path.join(DATASET_DIR, 'valid'), exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def create_dummy_or_custom_track(track_dir, sr=44100, duration_sec=10):
    os.makedirs(track_dir, exist_ok=True)
    num_samples = sr * duration_sec

    drums = torch.zeros((2, num_samples))
    bass = torch.zeros((2, num_samples))
    vocals = torch.zeros((2, num_samples))
    other = torch.randn((2, num_samples)) * 0.05

    t = torch.linspace(0, duration_sec, num_samples)
    flute_mono = 0.3 * torch.sin(2 * 3.14159 * 440 * t) + 0.15 * torch.sin(2 * 3.14159 * 880 * t)
    flute = torch.stack([flute_mono, flute_mono], dim=0)

    mixture = drums + bass + other + vocals + flute

    stems = {
        'drums': drums, 'bass': bass, 'other': other,
        'vocals': vocals, 'flute': flute, 'mixture': mixture
    }

    for name, audio in stems.items():
        sf.write(os.path.join(track_dir, f"{name}.wav"), audio.T.numpy(), sr)

create_dummy_or_custom_track(os.path.join(DATASET_DIR, 'train', 'track_001'))
create_dummy_or_custom_track(os.path.join(DATASET_DIR, 'valid', 'track_001'))
print("Directories and test dataset ready!")

Directories and test dataset ready!


In [11]:
#Define the FluteDemucsDataset Class
import torchaudio
from torch.utils.data import Dataset

class FluteDemucsDataset(Dataset):
    def __init__(self, root_dir, segment_sec=5.0, sample_rate=44100, augment=False):
        self.root_dir = root_dir
        self.segment_samples = int(segment_sec * sample_rate)
        self.sample_rate = sample_rate
        self.augment = augment
        self.stems = ['drums', 'bass', 'other', 'vocals', 'flute']

        self.tracks = [
            os.path.join(root_dir, d) for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        ]

    def __len__(self):
        return len(self.tracks)

    def __getitem__(self, idx):
        track_path = self.tracks[idx]
        stem_tensors = []

        for stem in self.stems:
            wav_path = os.path.join(track_path, f"{stem}.wav")
            audio, sr = torchaudio.load(wav_path)

            if sr != self.sample_rate:
                resampler = torchaudio.transforms.Resample(sr, self.sample_rate)
                audio = resampler(audio)

            stem_tensors.append(audio)

        targets = torch.stack(stem_tensors, dim=0)

        total_samples = targets.shape[-1]
        if total_samples > self.segment_samples:
            max_start = total_samples - self.segment_samples
            start = torch.randint(0, max_start + 1, (1,)).item() if self.augment else 0
            targets = targets[:, :, start:start + self.segment_samples]
        else:
            pad = self.segment_samples - total_samples
            targets = torch.nn.functional.pad(targets, (0, pad))

        mixture = targets.sum(dim=0)
        return mixture, targets

In [12]:
#Build the 5-Stem Demucs Architecture
import copy
from demucs.pretrained import get_model
from demucs.htdemucs import HTDemucs

def build_5stem_htdemucs():
    bag = get_model('htdemucs')
    pretrained_4stem = bag.models[0]

    init_args, init_kwargs = copy.deepcopy(pretrained_4stem._init_args_kwargs)
    init_args = list(init_args)

    new_sources = ['drums', 'bass', 'other', 'vocals', 'flute']
    if len(init_args) > 0:
        init_args[0] = new_sources
    else:
        init_kwargs['sources'] = new_sources

    model_5stem = HTDemucs(*init_args, **init_kwargs)

    sd_4 = pretrained_4stem.state_dict()
    sd_5 = model_5stem.state_dict()

    vocal_idx = 3

    for key in sd_5.keys():
        if key in sd_4:
            if sd_5[key].shape == sd_4[key].shape:
                sd_5[key].copy_(sd_4[key])
            else:
                s4, s5 = sd_4[key].shape, sd_5[key].shape
                diff_dims = [dim for dim in range(len(s4)) if s4[dim] != s5[dim]]

                if len(diff_dims) == 1 and s4[diff_dims[0]] * 5 == s5[diff_dims[0]] * 4:
                    dim = diff_dims[0]
                    c_per_stem = s4[dim] // 4

                    idx_orig = [slice(None)] * len(s5)
                    idx_orig[dim] = slice(0, 4 * c_per_stem)
                    sd_5[key][tuple(idx_orig)].copy_(sd_4[key])

                    idx_vocal = [slice(None)] * len(s4)
                    idx_vocal[dim] = slice(vocal_idx * c_per_stem, (vocal_idx + 1) * c_per_stem)

                    idx_flute = [slice(None)] * len(s5)
                    idx_flute[dim] = slice(4 * c_per_stem, 5 * c_per_stem)
                    sd_5[key][tuple(idx_flute)].copy_(sd_4[key][tuple(idx_vocal)])

    model_5stem.load_state_dict(sd_5)
    return model_5stem

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = build_5stem_htdemucs().to(device)
print("5-stem model loaded onto GPU successfully!")

Downloading: "https://dl.fbaipublicfiles.com/demucs/hybrid_transformer/955717e8-8726e21a.th" to /root/.cache/torch/hub/checkpoints/955717e8-8726e21a.th


100%|██████████| 80.2M/80.2M [00:00<00:00, 164MB/s]


5-stem model loaded onto GPU successfully!


In [13]:
#Run Training Loop
import gc
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

gc.collect()
torch.cuda.empty_cache()

train_dataset = FluteDemucsDataset(os.path.join(DATASET_DIR, 'train'), segment_sec=5.0, augment=True)
valid_dataset = FluteDemucsDataset(os.path.join(DATASET_DIR, 'valid'), segment_sec=5.0, augment=False)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_dataset, batch_size=1, shuffle=False, num_workers=0)

head_params = []
decoder_params = []

for name, param in model.named_parameters():
    if any(k in name for k in ['decoder.3', 'tdecoder.3']):
        param.requires_grad = True
        head_params.append(param)
    elif any(k in name for k in ['decoder', 'tdecoder']):
        param.requires_grad = True
        decoder_params.append(param)
    else:
        param.requires_grad = False

optimizer = torch.optim.AdamW([
    {'params': decoder_params, 'lr': 5e-5},
    {'params': head_params, 'lr': 3e-4}
], weight_decay=1e-4)

criterion = nn.L1Loss()
EPOCHS = 15
ACCUM_STEPS = 2
best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]")
    for step, (mixture, targets) in enumerate(pbar):
        mixture = mixture.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        estimates = model(mixture)

        loss_all = criterion(estimates, targets)
        loss_flute = criterion(estimates[:, 4], targets[:, 4])
        loss = (loss_all + 0.5 * loss_flute) / ACCUM_STEPS

        loss.backward()

        step_loss_val = loss.item() * ACCUM_STEPS
        flute_loss_val = loss_flute.item()
        train_loss += step_loss_val

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], max_norm=5.0
            )
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        pbar.set_postfix({'loss': f"{step_loss_val:.4f}", 'flute_l1': f"{flute_loss_val:.4f}"})

    avg_train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for mixture, targets in valid_loader:
            mixture = mixture.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            estimates = model(mixture)
            val_loss += criterion(estimates[:, 4], targets[:, 4]).item()

    avg_val_loss = val_loss / len(valid_loader)
    print(f"Epoch {epoch} | Train Loss: {avg_train_loss:.4f} | Val Flute L1: {avg_val_loss:.4f}")

    gc.collect()
    torch.cuda.empty_cache()

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        ckpt_path = os.path.join(CHECKPOINT_DIR, 'htdemucs_5stem_flute_best.pt')
        torch.save({
            'state_dict': model.state_dict(),
            'sources': model.sources,
            'epoch': epoch,
            'val_loss': best_val_loss
        }, ckpt_path)
        print(f"Saved new best checkpoint to {ckpt_path}")

Epoch 1/15 [Train]:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 0.0515 | Val Flute L1: 0.0043
Saved new best checkpoint to /content/drive/MyDrive/Demucs_Flute_Project/checkpoints/htdemucs_5stem_flute_best.pt


Epoch 2/15 [Train]:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 2 | Train Loss: 0.0431 | Val Flute L1: 0.0042
Saved new best checkpoint to /content/drive/MyDrive/Demucs_Flute_Project/checkpoints/htdemucs_5stem_flute_best.pt


Epoch 3/15 [Train]:   0%|          | 0/1 [00:00<?, ?it/s]

...

Epoch 13 | Train Loss: 0.0409 | Val Flute L1: 0.0037
Saved new best checkpoint to /content/drive/MyDrive/Demucs_Flute_Project/checkpoints/htdemucs_5stem_flute_best.pt


Epoch 14/15 [Train]:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 14 | Train Loss: 0.0406 | Val Flute L1: 0.0036
Saved new best checkpoint to /content/drive/MyDrive/Demucs_Flute_Project/checkpoints/htdemucs_5stem_flute_best.pt


Epoch 15/15 [Train]:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 15 | Train Loss: 0.0404 | Val Flute L1: 0.0035
Saved new best checkpoint to /content/drive/MyDrive/Demucs_Flute_Project/checkpoints/htdemucs_5stem_flute_best.pt
